# Mobile Price Prediction End-to-End Project

## 1. Introduction
This project aims to predict the price of mobile phones based on various specifications (features) such as RAM, Battery, Camera, etc. using Machine Learning regression techniques.

**Dataset:** The dataset includes features like `weight`, `resolution`, `ppi`, `cpu core`, `ram`, `battery`, etc.
**Target Variable:** `Price`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

## 2. Data Loading

In [ ]:
try:
    df = pd.read_csv('Cellphone.csv')
    print("Dataset loaded successfully.")
    print(f"Shape: {df.shape}")
except FileNotFoundError:
    print("Error: 'Cellphone.csv' not found. Please verify the file path.")

## 3. Exploratory Data Analysis (EDA)
We will inspect the data structure, check for missing values, and visualize distributions.

In [ ]:
# View first few rows
df.head()

In [ ]:
# Data info (types and non-null counts)
df.info()

In [ ]:
# Statistical summary
df.describe()

### 3.1 Check for Missing Values & Duplicates

In [ ]:
print("Missing Values:\n", df.isnull().sum())
print("\nDuplicates:\n", df.duplicated().sum())

### 3.2 Visualizations

In [ ]:
# Distribution of Target Variable 'Price'
plt.figure(figsize=(10, 6))
sns.histplot(df['Price'], kde=True, bins=30)
plt.title('Distribution of Price')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 10))
corr = df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Matrix')
plt.show()

## 4. Data Preprocessing
Steps:
1. Drop unnecessary columns (e.g., `Product_id`).
2. Check Skewness.
3. Treat Outliers.
4. Scale Features.

In [ ]:
# Drop Product_id as it is an identifier
if 'Product_id' in df.columns:
    df = df.drop('Product_id', axis=1)
    print("Dropped 'Product_id' column.")

In [ ]:
# Check Skewness
numeric_features = df.select_dtypes(include=[np.number]).columns
skewness = df[numeric_features].skew()
print("Skewness of features:\n", skewness.sort_values(ascending=False))

### 4.1 Outlier Treatment (IQR Method)

In [ ]:
def remove_outliers_iqr(data, columns):
    df_clean = data.copy()
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # Cap outlines or remove them. Here we verify count first.
        outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]
        if len(outliers) > 0:
            print(f"{col}: {len(outliers)} outliers detected. Removing...")
            df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    return df_clean

# Apply outlier removal
df_cleaned = remove_outliers_iqr(df, numeric_features)
print(f"\nShape after outlier removal: {df_cleaned.shape}")

### 4.2 Feature Scaling

In [ ]:
X = df_cleaned.drop('Price', axis=1)
y = df_cleaned['Price']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

X_scaled_df.head()

## 5. Model Development
We will split the data into training and testing sets and train four regression models:
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. ElasticNet Regression

In [ ]:
# Split Data
X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.2, random_state=42)
print(f"Training Set: {X_train.shape}")
print(f"Testing Set: {X_test.shape}")

### 5.1 Training and Evaluation Loop

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5)
}

results_list = []

for name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    mae = mean_absolute_error(y_test, y_test_pred)
    
    results_list.append({
        "Model": name,
        "Train R2": train_r2,
        "Test R2": test_r2,
        "RMSE": rmse,
        "MAE": mae
    })

results_df = pd.DataFrame(results_list)
results_df 

## 6. Conclusion and Comparison
Below is the comparison of model performance metrics.

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='Test R2', data=results_df)
plt.title('Model R2 Score Comparison')
plt.ylim(0, 1)
plt.show()

In [ ]:
print("Final Model Performance Sorted by Test R2:")
print(results_df.sort_values(by='Test R2', ascending=False))

------